# P06 · tendon vibration × polarity, 29-09-2026

**Why this session has its own notebook.** P06 is the **only participant whose vibration session
was run in both polarities**. P03 and P04 were cathodic only, so their vibration blocks sit in
`P03_P04_P06_tendon_vibration.ipynb`, where P06's cathodic recordings join them for a
three-participant comparison. Here the question is different and it can only be asked of P06:

> **Does tendon vibration act the same way on cathodic and on anodic stimulation?**

If the two polarities recruit different fibres — which the polarity notebooks suggest they might —
then a manipulation that works through **Ia afferents** should not touch them equally. Tendon
vibration is exactly that manipulation: it produces presynaptic inhibition of Ia terminals, so a
Ia-mediated response should be suppressed while a directly evoked one should not.

**The design.** 27 recordings: 3 blocks × 3 polarities × 3 protocols.

| | |
|---|---|
| **blocks** | Baseline → VIB off (vibrator strapped on, not running) → VIB ON |
| **polarities** | cathodic (*negative* in the log), anodic (*positive*), biphasic |
| **protocols** | 1 Hz burst, 30 Hz burst, ARC-EX |
| **vibrator** | **left wrist extensor** — the same site as P03 |

**VIB off is the control that matters.** The vibrator is attached but silent, so whatever it shows
is drift between blocks — time, electrode contact, the participant settling. A VIB ON effect has
to beat that, not just beat Baseline.

**What to look at.** `Ext. digitorum (L)` is the vibrated muscle; the other three are the internal
control. A real effect is VIB ON differing from Baseline **at the vibrated muscle and not at the
others**, by more than VIB off drifts.

**The 1 Hz blocks are the extra prize here.** Nobody else has them. At 1 s between pulses there is
no post-activation depression left, so the 1 Hz response is the cleanest single-pulse measure in
the whole dataset — and comparing it against 30 Hz in the same block is a genuine frequency
comparison, which the 30 Hz vs ARC-EX one is not (those are both 30 Hz).

**Colours:** 30 Hz **grey**, ARC-EX **red**, 1 Hz **blue**; plain = cathodic, hatched = anodic.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid,
                       fig_group_summary, fig_group_panels)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
# P06, 29-09-2026. That folder ALSO holds a copy of the whole 28-09 session, so only the files
# stamped 20260929 belong here - every one below is named explicitly.
D = "tSCS_CHUV_data/29-09-2026/"

ALL = {   # (protocol, polarity, block) -> csv.  "negative" in the log = cathodic, "positive" = anodic
 ("1 Hz",   "cathodic", "Baseline"): D + "Burst_1Hz_autosave_20260929_154415_417ms.csv",       # 15:44
 ("1 Hz",   "cathodic", "VIB off"):  D + "Burst_1Hz_autosave_20260929_162950_616ms.csv",       # 16:29
 ("1 Hz",   "cathodic", "VIB ON"):   D + "Burst_1Hz_autosave_20260929_180111_780ms.csv",       # 18:01
 ("1 Hz",   "anodic",   "Baseline"): D + "Burst_1Hz_autosave_20260929_154124_545ms.csv",       # 15:41
 ("1 Hz",   "anodic",   "VIB off"):  D + "Burst_1Hz_autosave_20260929_162709_372ms.csv",       # 16:27
 ("1 Hz",   "anodic",   "VIB ON"):   D + "Burst_1Hz_autosave_20260929_165059_014ms.csv",       # 16:50
 ("1 Hz",   "biphasic", "Baseline"): D + "Burst_1Hz_Biphasic_autosave_20260929_154957_624ms.csv",
 ("1 Hz",   "biphasic", "VIB off"):  D + "Burst_1Hz_Biphasic_autosave_20260929_163227_227ms.csv",
 ("1 Hz",   "biphasic", "VIB ON"):   D + "Burst_1Hz_Biphasic_autosave_20260929_172408_725ms.csv",
 ("30 Hz",  "cathodic", "Baseline"): D + "Burst_autosave_20260929_155626_234ms.csv",           # 15:56
 ("30 Hz",  "cathodic", "VIB off"):  D + "Burst_autosave_20260929_163809_190ms.csv",           # 16:38
 ("30 Hz",  "cathodic", "VIB ON"):   D + "Burst_autosave_20260929_175220_421ms.csv",           # 17:52
 ("30 Hz",  "anodic",   "Baseline"): D + "Burst_autosave_20260929_155345_812ms.csv",           # 15:53
 ("30 Hz",  "anodic",   "VIB off"):  D + "Burst_autosave_20260929_163517_916ms.csv",           # 16:35
 ("30 Hz",  "anodic",   "VIB ON"):   D + "Burst_autosave_20260929_170919_038ms.csv",           # 17:09
 ("30 Hz",  "biphasic", "Baseline"): D + "Burst_Biphasic_autosave_20260929_155921_927ms.csv",
 ("30 Hz",  "biphasic", "VIB off"):  D + "Burst_Biphasic_autosave_20260929_164040_213ms.csv",
 ("30 Hz",  "biphasic", "VIB ON"):   D + "Burst_Biphasic_autosave_20260929_171716_204ms.csv",
 ("ARC-EX", "cathodic", "Baseline"): D + "Modulated_autosave_20260929_160545_334ms.csv",       # 16:05
 ("ARC-EX", "cathodic", "VIB off"):  D + "Modulated_autosave_20260929_164440_099ms.csv",       # 16:44
 ("ARC-EX", "cathodic", "VIB ON"):   D + "Modulated_autosave_20260929_170027_460ms.csv",       # 17:00
 ("ARC-EX", "anodic",   "Baseline"): D + "Modulated_autosave_20260929_160150_049ms.csv",       # 16:01
 ("ARC-EX", "anodic",   "VIB off"):  D + "Modulated_autosave_20260929_164216_867ms.csv",       # 16:42
 ("ARC-EX", "anodic",   "VIB ON"):   D + "Modulated_autosave_20260929_173301_845ms.csv",       # 17:33
 ("ARC-EX", "biphasic", "Baseline"): D + "Modulated_Biphasic_autosave_20260929_160823_040ms.csv",
 ("ARC-EX", "biphasic", "VIB off"):  D + "Modulated_Biphasic_autosave_20260929_164704_430ms.csv",
 ("ARC-EX", "biphasic", "VIB ON"):   D + "Modulated_Biphasic_autosave_20260929_174216_620ms.csv",
}

# ======== what to analyse ========
PROTOCOLS  = ["30 Hz", "ARC-EX"]              # add "1 Hz" for the frequency comparison
POLARITIES = ["cathodic", "anodic"]           # add "biphasic" for all three
# =================================

CONDITIONS = ["Baseline", "VIB off", "VIB ON"]          # the bars inside every panel
PANELS     = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]   # one panel each
RUN        = {(f"{p} · {pol}", b): ALL[(p, pol, b)]
              for p in PROTOCOLS for pol in POLARITIES for b in CONDITIONS}

VIBRATED = "Ext. digitorum (L)"      # the vibrator sat on the left wrist extensor
SITE     = "left wrist extensor"

# colour = protocol, fill pattern = polarity, shade = block
PCOL   = {"1 Hz": "#2E6F95", "30 Hz": "#9A9A9A", "ARC-EX": "#C0392B"}
SHADE  = {"Baseline": 0.45, "VIB off": 0.75, "VIB ON": 1.0}   # pale -> solid
def _mix(c, f):
    import matplotlib.colors as mc
    r, g, b = mc.to_rgb(c)
    return tuple(1 - (1 - v) * f for v in (r, g, b))
COND_COLOUR = {c: _mix("#9A9A9A", SHADE[c]) for c in CONDITIONS}

MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for b in CONDITIONS:
        f = runs[(pan, b)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:20s} {b:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA, "
              f"step {amps[1] - amps[0]:g}")

## 2 · The motor thresholds you chose

Read straight from `results/P06_2026-09-29_tendon_vibration/mt_*.csv`. One row per recording, one
column per muscle: every intensity stacked, **your threshold in red**, the mA in the corner.
`not picked` = nothing saved yet — pick them in section 5.

Each panel's **Baseline** recording sets the intensity that panel is analysed at, so vibration
cannot move the intensity as well as the response. Every polarity and protocol carries its own
thresholds; they are never forced onto one value.

In [ ]:
CHECK = {f"{pan} · {b}": runs[(pan, b)] for pan in PANELS for b in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title="P06 · 29-09 — the thresholds saved for each recording")

### The numbers, and where each one came from

In [ ]:
MT = {}      # MT[panel] = {muscle: mA}, from that panel's own BASELINE recording
for pan in PANELS:
    csv = runs[(pan, "Baseline")]
    MT[pan] = thresholds_for(csv, MUSCLES, consecutive=2, **KW)
    print(f"{pan:20s} Baseline   {threshold_source(csv)}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}

print("\nmotor threshold (mA), as picked in each recording")
print(f"{'recording':26s}" + "".join(f"{m.split(' (')[0][:9] + m[-3:]:>16s}" for m in MUSCLES))
for lab in CHECK:
    print(f"{lab:26s}" + "".join(f"{str(MT_ALL[lab].get(m) or '-'):>16s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2, showing how the peaks are found at the threshold picked in **that**
recording: the 10 pulses re-aligned on their own onset, with the max (v) and min (^) the
peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window |
| **red bands** | the only places a peak may be taken, `ANCHOR_WIN_MS` either side of the train's own average latency |
| red ring | flagged — on a window border, or at a different latency |
| corner | `clean`, or how many of the 10 pulses were flagged |

**At 1 Hz there is only one pulse per second**, so a 1 Hz recording has a single response per
train and the train-shape measures do not apply to it — check it here, read it in section 4 as a
1st-pulse amplitude only.

`TRY` loosens the settings **for this figure only**; copy anything you keep into `KW` in
section 1.

In [ ]:
TRY = dict(KW)
# TRY["anchor_win_ms"] = 6.0
# TRY["anchor"] = None
# TRY["resp_start_ms"] = 6.0
# TRY["min_snr"] = None

fig_detection_grid(CHECK, MT_ALL, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title="P06 · 29-09 — peak detection at each recording's own threshold", **TRY)

## 4 · The results

### 4a · The vibration effect, one panel per protocol and polarity

Every block as a % of its **own panel's Baseline**, read at the Baseline threshold. `*` marks the
vibrated muscle. **100 % = unchanged.**

**The comparison this notebook exists for** is cathodic against anodic *within* a protocol: if
vibration suppresses one polarity and not the other, the two are not recruiting the same thing.
Read VIB ON against VIB off, not against Baseline alone.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, "Baseline") for pan in PANELS])

tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base="Baseline", match="common", **KW)
fig_vibration(tab, PANELS, MUSCLES, CONDITIONS, colours=COND_COLOUR,
              vibrated={pan: VIBRATED for pan in PANELS},
              ylabel="% of that panel's baseline", ylim=(0, 200),
              titles={pan: f"{pan}   (vibrator on the {SITE})" for pan in PANELS},
              title="P06 · 29-09 — tendon vibration, by protocol and polarity")

### The numbers, so they can be checked

In [ ]:
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}" + "".join(f"{c:>12s}" for c in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        row = [tab.get((pan, c, m), {}).get("pct", float("nan")) for c in CONDITIONS]
        if not any(v == v for v in row):
            continue
        amp = next((tab[(pan, c, m)]["amp"] for c in CONDITIONS if (pan, c, m) in tab), None)
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}"
              + "".join(f"{(f'{v:.0f} %' if v == v else '-'):>12s}" for v in row))

### 4b · Cathodic against anodic, the same effect side by side

Section 4a has one panel per protocol *and* polarity, which is four panels to compare by eye.
This re-slices the same numbers into the comparison the notebook is for: **one panel per protocol,
one bar per polarity**, so cathodic and anodic sit next to each other in the same muscle.

Two figures, **VIB off** and **VIB ON**, both as a % of their own panel's Baseline. Read them as a
pair: VIB off is the drift with the vibrator attached but silent, VIB ON is drift plus vibration.
**A vibration effect is VIB ON departing from Baseline by more than VIB off does, at the vibrated
muscle.**

There is one participant here, so these are single measurements — no error bars, nothing averaged.
The dots and SDs of the group figures elsewhere do not apply.

In [ ]:
# the same table, re-keyed so polarity is the bar and protocol is the panel
for block in ("VIB off", "VIB ON"):
    sliced = {(p, pol, m): tab[(f"{p} · {pol}", block, m)]
              for p in PROTOCOLS for pol in POLARITIES for m in MUSCLES
              if (f"{p} · {pol}", block, m) in tab}
    for key, lab in (("pct", "1st pulse"), ("pct_all", "all 10 pulses")):
        fig_vibration(sliced, PROTOCOLS, MUSCLES, POLARITIES, key=key,
                      colours={"cathodic": "#9A9A9A", "anodic": "#C0392B"},
                      hatches={"cathodic": "", "anodic": "///"},
                      vibrated={p: VIBRATED for p in PROTOCOLS},
                      ylabel=f"% of baseline", ylim=(0, 200),
                      titles={p: p for p in PROTOCOLS},
                      title=f"{block}, % of baseline — {lab}",
                      save=None)

### 4c · One figure per muscle, with the EMG

The 1st pulse in mV, the 2nd and the mean of 2–10 as a % of it, and the EMG underneath.
Millivolts *are* comparable here — one participant, one session, one electrode.

In [ ]:
MT_cond = {(pan, c): {m: tab[(pan, c, m)]["amp"] for m in MUSCLES if (pan, c, m) in tab}
           for pan in PANELS for c in CONDITIONS}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(CONDITIONS), steps=0,
             colours=COND_COLOUR, names={c: c for c in CONDITIONS},
             linestyles={"Baseline": "-", "VIB off": (0, (4, 2)), "VIB ON": (0, (1, 1))},
             min_snr_ratio=2.0, rest_from=2, with_p1=True, traces=True,
             title=f"{m}{'  *vibrated' if m == VIBRATED else ''}", save=None, **KW)

## 5 · Change a threshold

One cell per recording: move the slider, press **Set to slider**, then **Save**. Re-run section 2
and every figure follows.

> Only a **Baseline** pick changes section 4 — every block is read at its panel's baseline
> threshold. Re-picking a VIB block changes section 2 and 3 only.

In [ ]:
def repick(key):
    """Open the picker for one recording, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('30 Hz · cathodic', 'Baseline'))

In [ ]:
repick(('30 Hz · cathodic', 'VIB off'))

In [ ]:
repick(('30 Hz · cathodic', 'VIB ON'))

In [ ]:
repick(('30 Hz · anodic', 'Baseline'))

In [ ]:
repick(('30 Hz · anodic', 'VIB off'))

In [ ]:
repick(('30 Hz · anodic', 'VIB ON'))

In [ ]:
repick(('ARC-EX · cathodic', 'Baseline'))

In [ ]:
repick(('ARC-EX · cathodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · cathodic', 'VIB ON'))

In [ ]:
repick(('ARC-EX · anodic', 'Baseline'))

In [ ]:
repick(('ARC-EX · anodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · anodic', 'VIB ON'))